# ndgpu — GPU validation & benchmark (Google Colab)

Validates the CUDA path of the ndgpu neutron diffusion/SP3 solver on a free Colab GPU.

1. **Runtime → Change runtime type → T4 GPU**, then run all cells.
2. When prompted, upload `dist/ndgpu-src.zip` from the repo.

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload dist/ndgpu-src.zip
zip_name = next(iter(uploaded))
%pip install -q {zip_name}
try:
    import cupy
except ImportError:
    %pip install -q cupy-cuda12x
!nvidia-smi -L

## Correctness on GPU: analytic bare reactor + CPU/GPU agreement

In [ ]:
from ndgpu import DiffusionEigenSolver, SP3EigenSolver, Grid, PWR_TWO_GROUP, k_bare_box, k_bare_box_sp3

size = (150.0, 150.0, 150.0)
grid = Grid(shape=(96, 96, 96), size=size)
for cls, ref in [(DiffusionEigenSolver, k_bare_box), (SP3EigenSolver, k_bare_box_sp3)]:
    res = cls(grid, PWR_TWO_GROUP, device='gpu').solve()
    exact = ref(PWR_TWO_GROUP, size)
    dpcm = (res.k_eff - exact) * 1e5
    print(f'{cls.__name__:22s} {res}')
    print(f'{"":22s} vs exact analytic k: {dpcm:+.1f} pcm')
    assert abs(dpcm) < 20, 'GPU result disagrees with analytic solution'
print('\nGPU path validated against analytic solutions.')

## CPU vs GPU speedup

In [ ]:
from ndgpu import DiffusionEigenSolver, Grid, PWR_TWO_GROUP

print(f"{'grid':>8} {'unknowns':>12} {'cpu [s]':>9} {'gpu [s]':>9} {'speedup':>8}")
for n in (64, 96, 128, 160):
    grid = Grid(shape=(n, n, n), size=(150.0, 150.0, 150.0))
    t = {}
    for dev in ('cpu', 'gpu'):
        solver = DiffusionEigenSolver(grid, PWR_TWO_GROUP, device=dev)
        solver.solve(max_outer=3, tol_k=0)  # warm-up, not timed
        t[dev] = solver.solve().solve_seconds
    print(f'{n:>6}^3 {2 * grid.n_cells:>12,} {t["cpu"]:>9.2f} {t["gpu"]:>9.2f} {t["cpu"] / t["gpu"]:>7.1f}x')

## C5G7 benchmark on GPU

In [ ]:
from ndgpu import DiffusionEigenSolver, SP3EigenSolver
from ndgpu.benchmarks import K_REFERENCE_2D, build_c5g7_2d

prob = build_c5g7_2d(cells_per_pin=4)
for cls in (DiffusionEigenSolver, SP3EigenSolver):
    solver = cls(prob.grid, prob.materials, prob.material_map, bc=prob.bc, device='gpu')
    res = solver.solve(tol_k=1e-6, tol_source=1e-5)
    print(f'{cls.__name__:22s} k={res.k_eff:.5f} '
          f'({(res.k_eff - K_REFERENCE_2D) * 1e5:+.0f} pcm vs transport)  {res.solve_seconds:.2f} s')